In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network
import os
import matplotlib.colors as mcolors

In [ ]:
folder_path='/Users/pipe/work_dir/observed_synchrony/paper_results_edge_degree_15_2025may29/results_edge_degree_15/fig9_fast_first_division/'
os.chdir(folder_path)

In [7]:
# Loading the graphs
fast_first_div=nx.read_graphml('/Users/pipe/work_dir/observed_synchrony/paper_results_edge_degree_15_2025may29/results_edge_degree_15/fig9_fast_first_division/fast_first_div/network_dir/1_0.graphml')
synchronous_div=nx.read_graphml('/Users/pipe/work_dir/observed_synchrony/paper_results_edge_degree_15_2025may29/results_edge_degree_15/fig9_fast_first_division/synchronized_strain/network_dir/1_0.graphml')

In [8]:
print(f"fast first divisions has {fast_first_div.number_of_nodes()} nodes.")
print(f"synchronized divisions has {synchronous_div.number_of_nodes()} nodes.")


fast first divisions has 930 nodes.
synchronized divisions has 260 nodes.


In [13]:

# Calculate node degrees for both networks
fast_first_div_degrees = dict(fast_first_div.degree())
synchronous_div_degrees = dict(synchronous_div.degree())


# Function to identify filament nodes
def identify_filament_nodes(graph, min_length=3):
    filament_nodes = set()
    visited = set()

    def traverse_filament(node):
        path = [node]
        current = node
        visited.add(current)

        while True:
            neighbors = list(graph.neighbors(current))
            unvisited_neighbors = [n for n in neighbors if n not in visited]

            if len(unvisited_neighbors) == 1 and graph.degree(unvisited_neighbors[0]) <= 2:
                next_node = unvisited_neighbors[0]
                path.append(next_node)
                visited.add(next_node)
                current = next_node
            else:
                break

        return path if len(path) >= min_length else []

    for node, degree in dict(graph.degree()).items():
        if degree == 1 and node not in visited:
            filament = traverse_filament(node)
            filament_nodes.update(filament)

    return filament_nodes

# Function to create and save network visualization
def create_network_visualization(graph, degrees, filename, node_size, edge_width, min_filament_length=3):
    nt = Network('1000px', '1000px', notebook=True)
    nt.from_nx(graph)
    
    
    # Identify filament nodes
    filament_nodes = identify_filament_nodes(graph, min_filament_length)
    
    for node in nt.nodes:
        node_id = node['id']
        node['size'] = node_size
        node['label'] = ''
        if node_id in filament_nodes:
            node['color'] = 'red'
        else:
            node['color'] = 'blue'
    
    for edge in nt.edges:
        if edge['from'] in filament_nodes and edge['to'] in filament_nodes:
            edge['color'] = 'red'
        else:
            edge['color'] = 'blue'
        edge['width'] = edge_width
    
    nt.toggle_hide_edges_on_drag(True)
    nt.show_buttons(filter_=['physics'])
    nt.set_edge_smooth('dynamic')
    nt.save_graph(filename)

# Create visualizations for both networks
min_fil_size=3
create_network_visualization(fast_first_div, fast_first_div_degrees, 'fast_first_div_filament_size_'+str(min_fil_size)+'.html', node_size=36, edge_width=18)
create_network_visualization(synchronous_div, synchronous_div_degrees, 'synchronous_div_filament_size_'+str(min_fil_size)+'.html', node_size=36, edge_width=18)